In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-stage-3-2026")

print("Path to dataset files:", path)

In [ ]:
# Write your code here
import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
from tqdm import tqdm

In [ ]:
from IPython.display import clear_output
!pip install torch torchvision matplotlib scikit-learn
clear_output()

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu") # it is better to run CNNs with GPUs for faster computation
device

In [ ]:
# Write your code here
import os
import glob
from torch.utils.data import Dataset
from PIL import Image
import os
from torchvision.datasets import ImageFolder
from torchvision import transforms

train_transform = transforms.Compose([
    transforms.RandomRotation(15),
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])
test_transform = transforms.Compose([
    transforms.RandomRotation(15),
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])
# train_dir=("/kaggle/input/q1-stage-3-2026/PlantVillage/train")
# test_dir=("/kaggle/input/q1-stage-3-2026/PlantVillage/test")

from torchvision.datasets import ImageFolder
# train_dataset = ImageFolder(root=train_dir, transform=train_transform)

# test_dataset  = ImageFolder(root=test_dir,  transform=test_transform)

# train_dataset[4][0].max()

In [ ]:
class FashionMNISTCustom(Dataset):

    def __init__(self, root, train: bool, transform=None, download: bool = True):
        super().__init__()
        self.base = datasets.FashionMNIST(root=root, train=train, download=download)
        self.transform = transform

    def __len__(self):
        return len(self.base)

    def __getitem__(self, idx):
        img, label = self.base[idx]  # img is a PIL image
        if self.transform is not None:
            img = self.transform(img)
        return {"image": img, "label": int(label), "index": int(idx)}


In [ ]:
train_ds = FashionMNISTCustom(root="./data", train=True,  download=True, transform=train_transform)
test_ds  = FashionMNISTCustom(root="./data", train=False, download=True, transform=test_transform)

# We got the label information from the Fashion-MNIST dataset page on Kaggle
class_names = ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat", "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]



In [ ]:
batch_size = 128 # design choice: you could change this to 16, 32, 64...

train_loader = DataLoader(
    train_ds,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

test_loader = DataLoader(
    test_ds,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

In [ ]:
batch = next(iter(train_loader))
xb = batch["image"]
yb = batch["label"]
print("Train loader batch sample X shape (input):", tuple(xb.shape), "\Train loader batch sample y shape (target):", tuple(yb.shape))


In [ ]:
fig, axes = plt.subplots(2, 6, figsize=(10, 4))
for ax in axes.ravel():
    idx = np.random.randint(0, len(train_ds))
    sample = train_ds[idx]
    x, y = sample["image"], sample["label"]

    # unnormalize for display (approx)
    x_vis = (x * 0.5) + 0.5
    ax.imshow(x_vis.squeeze(0), cmap="gray")
    ax.set_title(class_names[y], fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# Write your code here
class SimpleFashionCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(

            # TRACE THE SHAPE OF THE TENSORS AS IT PASSES THROUGH THE CONV LAYERS TO AVOID SHAPE MISMATCH ERRORS
            # HOW? --> by using the output features formula shown above

            nn.Conv2d(1, 32, kernel_size=3, padding=1),  # [B,32,28,28]
            nn.ReLU(),
            nn.MaxPool2d(2),                             # [B,32,14,14]

            # TO-DO: Add second Conv2d layer (input: 32, output: 64, kernel: 3, padding: 1)
            nn.Conv2d(32, 64, kernel_size=3, padding=1),  # [B,64,14,14]
            nn.ReLU(),
            # TO-DO: Add MaxPool2d with kernel_size=2
            nn.MaxPool2d(2),                            # [B,64,7,7]


            # TO-DO: Add second Conv2d layer (input: 32, output: 64, kernel: 3, padding: 1)
            nn.Conv2d(64, 128, kernel_size=3, padding=1),  # [B,64,14,14]
            nn.ReLU(),
            # TO-DO: Add MaxPool2d with kernel_size=2
            nn.MaxPool2d(2),

            # TO-DO: Add second Conv2d layer (input: 32, output: 64, kernel: 3, padding: 1)
            nn.Conv2d(128, 256, kernel_size=3, padding=1),  # [B,64,14,14]
            nn.ReLU(),
            # TO-DO: Add MaxPool2d with kernel_size=2
            nn.MaxPool2d(2),

            # TO-DO: Add second Conv2d layer (input: 32, output: 64, kernel: 3, padding: 1)
            nn.Conv2d(256, 512, kernel_size=3, padding=1),  # [B,64,14,14]
            nn.ReLU(),
            # TO-DO: Add MaxPool2d with kernel_size=2
            nn.MaxPool2d(2),
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            # TO-DO: Calculate input features (channels × height × width)
            # After 2 MaxPool2d(2): 28 → 14 → 7
            nn.Linear(512 * 7 * 7, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        # TO-DO: Pass x through features
        x = self.features(x)
        # TO-DO: Pass result through classifier
        x = self.classifier(x)
        return x


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu") # it is better to run CNNs with GPUs for faster computation
device

In [ ]:
model = SimpleFashionCNN().to(device)

model # let's have a look at our model architecture

In [ ]:
# Write your code here
def accuracy_from_logits(logits, labels):
    # TO-DO: Get predicted class indices
    # HINT: Use torch.argmax with dim=1
    preds = torch.argmax(logits, dim=1)
    # TO-DO: Calculate and return accuracy
    # HINT: Compare preds with labels, convert to float, mean, then .item()
    return (preds == labels).float().mean().item()

def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, total_acc = 0.0, 0.0

    for batch in tqdm(loader):
        images, labels = batch['image'], batch['label']
        images, labels = images.to(device), labels.to(device)

        # TO-DO: Zero the gradients
        optimizer.zero_grad()

        logits = model(images)
        loss = criterion(logits, labels)

        # TO-DO: Backward pass
        loss.backward()

        # TO-DO: Update parameters
        optimizer.step()

        total_loss += loss.item()
        total_acc += accuracy_from_logits(logits.detach(), labels)

    return total_loss / len(loader), total_acc / len(loader)


In [ ]:
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, total_acc = 0.0, 0.0

    with torch.no_grad():
        for batch in tqdm(loader):
            images, labels = batch['image'], batch['label']
            images, labels = images.to(device), labels.to(device)

            # TO-DO: Get model predictions
            logits = model(images)

            # TO-DO: Calculate loss
            # HINT: Use the criterion function
            loss = criterion(logits, labels)

            total_loss += loss.item()
            total_acc += accuracy_from_logits(logits, labels)

    return total_loss / len(loader), total_acc / len(loader)


In [ ]:
# Write your code here
# Training setup
# TO-DO: Define loss criterion
# HINT: What loss function works for multi-class classification?
criterion = nn.CrossEntropyLoss()

# TO-DO: Set learning rate
learning_rate = 0.001  # Typical range: 0.0001 to 0.01

# TO-DO: Define optimizer
# HINT: Pass model.parameters() and learning_rate
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# TO-DO: Set number of epochs
num_epochs = 10  # How many times to iterate through the dataset?

# Initialize history tracking
history = {"train_loss": [], "train_acc": [], "test_loss": [], "test_acc": []}

# Training loop
for epoch in range(num_epochs):
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion)
    test_loss, test_acc = evaluate(model, test_loader, criterion)

    # Store history
    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["test_loss"].append(test_loss)
    history["test_acc"].append(test_acc)

    print(f'Epoch {epoch+1}/{num_epochs} - Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}, Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.4f}')


In [ ]:
plt.figure(figsize=(15, 5))

plt.subplot(1, 2, 1)
plt.plot(range(1, num_epochs+1), history["train_loss"], label="train_loss", marker='o')
plt.plot(range(1, num_epochs+1), history["test_loss"], label="test_loss", marker='o')
plt.xlabel("epoch")
plt.ylabel("loss")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(range(1, num_epochs+1), history["train_acc"], label="train_acc", marker='o')
plt.plot(range(1, num_epochs+1), history["test_acc"], label="test_acc", marker='o')
plt.xlabel("epoch")
plt.ylabel("accuracy")
plt.legend()

plt.show()

In [ ]:
# Write your code here
